In [2]:
import os
import pdfplumber
from PIL import Image
import base64
from io import BytesIO
from openai import OpenAI, AsyncOpenAI
import yaml
import asyncio
from portkey_ai import Portkey , AsyncPortkey

In [3]:
# client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))                       # Initialize the OpenAI Async client

client = Portkey(base_url=os.environ.get("PORTKEY_BASE_URL"), api_key=os.environ.get("PORTKEY_API_KEY") )

In [4]:
sample_pdf_path=r"..\..\Sample PDFs\Blueprint.pdf"  
os.makedirs("pdf_markdowns", exist_ok=True)              # Create a folder to save markdown files
folder_name="pdf_markdowns"                              # Folder to save markdown files
pdf_name=os.path.basename(sample_pdf_path)[:-4] + ".md"  # Name of the markdown file to save results

In [5]:
with open("prompts.yaml",'r') as file:                                         # Load the prompt from a YAML file                          
    prompt = yaml.safe_load(file)

In [6]:
prompt=prompt['prompt']

In [7]:
def pdf_to_images_in_list(pdf_path):

    """Convert each page of a PDF file to a list of PIL Images.
    Args:
        pdf_path (str): Path to the PDF file.
    Returns:
        list: A list of PIL Image objects, each representing a page of the PDF.
    """
    images = []
    max_pages = 12
    # Open the PDF file using pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        # for page in pdf.pages:
        #     # Render the page as an image
        #     pil_image = page.to_image(resolution=360).original
            
        #     # Append the PIL Image to the list
        #     images.append(pil_image)

        for i, page in enumerate(pdf.pages):
            if i >= max_pages:   # stop after N pages
                break

            pil_image = page.to_image(resolution=360).original
            images.append(pil_image)
    
    return images

In [8]:
def pil_to_base64(image: Image.Image, format="PNG") -> str:
    """
    Convert a PIL Image to a Base64 encoded string.
    Args:
        image (Image.Image): The PIL Image to convert.
        format (str): The format to save the image in (default is "PNG").
        Returns:
        str: Base64 encoded string of the image.
    """
    buffered = BytesIO()
    image.save(buffered, format=format)  # Convert the image to bytes
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")  # Encode to Base64
    return img_str

In [9]:
def list_of_base64_images(pages_image_list):
    base64_image_list=[]
    for page_img in pages_image_list:
        base64_image_list.append(pil_to_base64(page_img))
    return base64_image_list

In [10]:
def message_template(prompt:str, img):
    """Create a message template for the OpenAI API request.
    Args:
        prompt (str): The prompt to be sent to the model.
        img (str): Base64 encoded image string.
    Returns:
        list: A list containing the message structure for the OpenAI API.
    """
    message = [{
    "role": "user",
    "content": [
        {"type": "text", "text": prompt},
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}}
    ] 
}]
    return message

In [11]:
def streaming_of_openai(client, message,model_name="@personal-openai/gpt-4o-mini"):
    stream = client.chat.completions.create(
            model=model_name,
            messages=message
        )
    return stream

In [12]:
async def stream_single_page(client, base64_img, prompt, model_name="@personal-openai/gpt-4o"):
    message = message_template(prompt, base64_img)
    stream = await client.chat.completions.create(
        model=model_name,
        messages=message
    )

    return stream.choices[0].message.content

In [13]:
def concatenate_results(results):
    """Concatenate results into a single string with page breaks."""
    whole_text = ""
    for page_num, result in enumerate(results):
        whole_text += f"## Start of Page {page_num + 1}\n\n{result}\n\n"
        
    return whole_text

In [14]:
def save_results_to_markdown(whole_text, folder_name,pdf_name):
    """Save the concatenated results to a Markdown file."""
    with open(os.path.join(folder_name,pdf_name), "w", encoding="utf-8") as f:
        f.write(whole_text)
    print(f"Results saved to {os.path.join(folder_name,pdf_name)}")

In [15]:
async def process_all_pages(base64_images, prompt):
    # client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    client = AsyncPortkey(base_url=os.environ.get("PORTKEY_BASE_URL"), api_key=os.environ.get("PORTKEY_API_KEY"))
    tasks = [
        stream_single_page(client, img, prompt)
        for img in base64_images
    ]
    responses = await asyncio.gather(*tasks)
    return responses

In [16]:
pdf_pages_images_list=pdf_to_images_in_list(sample_pdf_path)
base64_images_list=list_of_base64_images(pdf_pages_images_list)

In [17]:
results=await process_all_pages(base64_images_list, prompt)

In [18]:
save_results_to_markdown(concatenate_results(results=results), folder_name=folder_name,pdf_name=pdf_name)

Results saved to pdf_markdowns\Blueprint.md
